# Day 8: JavaScript and Async JavaScript
**Variables, arrays, objects, functions and callbacks, then promises, async and await, fetch and JSON**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Google Colab already has Node.js. In each section, `%%writefile` saves a JavaScript file and `!node` runs it.

**VS Code:** install Node.js LTS from nodejs.org, save each file, then run `node file.js` in the terminal.

In [ ]:
# ===== Option A: Google Colab (active) =====
!node --version

# ===== Option B: VS Code =====
# In the VS Code terminal:
#   node --version

## 2. Variables, types and template literals (Session 1)

In [ ]:
%%writefile basics.js
// Variables
const city = "Pune";
let count = 3;
count += 1;
console.log(`${city} has ${count} open tickets`);

// Types
console.log(typeof count, typeof city, typeof true, typeof undefined, typeof null, typeof [1, 2]);

// Functions preview
const toF = (c) => c * 9 / 5 + 32;
const isEven = (n) => n % 2 === 0;
console.log(`37 C = ${toF(37).toFixed(1)} F`);
console.log(`7 is ${isEven(7) ? "even" : "odd"}`);

// Gotchas
console.log("5" == 5, "5" === 5);
console.log(0.1 + 0.2, (0.1 + 0.2).toFixed(2));

In [ ]:
# ===== Option A: Google Colab (active) =====
!node basics.js

# ===== Option B: VS Code =====
# node basics.js

## 3. Arrays, objects and loops with SmartDesk tickets

In [ ]:
%%writefile tickets.js
const tickets = [
  { id: "T-101", title: "No signal at home", priority: "high", ageHours: 20, status: "open" },
  { id: "T-102", title: "Double charge", priority: "high", ageHours: 9, status: "open" },
  { id: "T-103", title: "Change plan", priority: "low", ageHours: 3, status: "closed" },
  { id: "T-104", title: "Slow 5G data", priority: "medium", ageHours: 30, status: "pending" },
  { id: "T-105", title: "SIM not activated", priority: "medium", ageHours: 12, status: "open" },
];

module.exports = { tickets };

// Only print when run directly (node tickets.js), not when required by another file
if (require.main === module) {
  const highOpen = tickets.filter(t => t.priority === "high" && t.status === "open").map(t => t.title);
  const open = tickets.filter(t => t.status === "open");
  const avgAge = open.reduce((sum, t) => sum + t.ageHours, 0) / open.length;
  const oldest = tickets.reduce((max, t) => (t.ageHours > max.ageHours ? t : max));
  const { id, ageHours } = oldest;

  console.log("High and open:", highOpen.join(", "));
  console.log("Average open age:", avgAge.toFixed(1), "h");
  console.log(`Oldest: ${id} (${ageHours} h)`);

  for (const t of tickets) {
    if (t.ageHours > 24) console.log("Escalate:", t.id);
  }
  for (const key in oldest) console.log(" ", key, "=", oldest[key]);

  const updated = { ...oldest, status: "escalated" };
  console.log(oldest.status, "->", updated.status);
}

In [ ]:
# ===== Option A: Google Colab (active) =====
!node tickets.js

# ===== Option B: VS Code =====
# node tickets.js

### TODO 1: Print the number of tickets per status using a `for...of` loop and an object as a counter.

In [ ]:
# TODO: write your code here (use %%writefile and !node)

## 4. Functions and callbacks (Session 2)

In [ ]:
%%writefile callbacks.js
const { tickets } = require("./tickets.js");

// Three ways to write a function
function greet(name = "friend") { return `Hello, ${name}!`; }
const square = function (n) { return n * n; };
const isUrgent = (t) => t.priority === "high";
console.log(greet(), greet("Aiko"), square(4));

// Your own function that takes a callback
function groupBy(items, keyFn) {
  return items.reduce((acc, item) => {
    (acc[keyFn(item)] ??= []).push(item);
    return acc;
  }, {});
}
const byStatus = groupBy(tickets, t => t.status);
const byAge = groupBy(tickets, t => (t.ageHours > 24 ? "old" : "new"));
const counts = (g) => Object.fromEntries(Object.entries(g).map(([k, v]) => [k, v.length]));
console.log(counts(byStatus), counts(byAge));

// Built-in functions that take callbacks
tickets.filter(isUrgent).forEach(t => console.log("Urgent:", t.id));
const byOldest = [...tickets].sort((a, b) => b.ageHours - a.ageHours).map(t => t.id);
console.log("Oldest first:", byOldest.join(" "));
setTimeout(() => console.log("Reminder: follow up on T-104"), 200);
console.log("This line prints before the reminder");

In [ ]:
# ===== Option A: Google Colab (active) =====
!node callbacks.js

# ===== Option B: VS Code =====
# node callbacks.js

## 5. The event loop: predict, then run (Session 3)

In [ ]:
%%writefile order.js
console.log("1: order taken");
setTimeout(() => console.log("4: coffee ready (timer 100 ms)"), 100);
setTimeout(() => console.log("3: tea ready (timer 0 ms)"), 0);
console.log("2: next customer");

In [ ]:
# ===== Option A: Google Colab (active) =====
!node order.js

# ===== Option B: VS Code =====
# node order.js

## 6. Promises

In [ ]:
%%writefile promises.js
function cook(dish, ms = 300) {
  return new Promise((resolve, reject) => {
    setTimeout(() => {
      if (dish === "pizza") reject(new Error(`out of cheese for ${dish}`));
      else resolve(`${dish} ready`);
    }, ms);
  });
}

cook("rice")
  .then(msg => { console.log(msg); return cook("curry"); })
  .then(msg => { console.log(msg); return cook("pizza"); })
  .then(msg => console.log(msg))
  .catch(err => console.log("Error:", err.message))
  .finally(() => console.log("Kitchen closed"));

In [ ]:
# ===== Option A: Google Colab (active) =====
!node promises.js

# ===== Option B: VS Code =====
# node promises.js

## 7. async / await and Promise.all

In [ ]:
%%writefile async_demo.js
const wait = (ms) => new Promise(resolve => setTimeout(resolve, ms));
const cook = async (dish) => { await wait(500); return `${dish} ready`; };

async function main() {
  let start = Date.now();
  await cook("rice");
  await cook("curry");
  await cook("dal");
  console.log(`Sequential: ${Date.now() - start} ms`);

  start = Date.now();
  const dishes = await Promise.all([cook("rice"), cook("curry"), cook("dal")]);
  console.log(`Parallel:   ${Date.now() - start} ms`, dishes);

  try {
    await Promise.reject(new Error("burnt toast"));
  } catch (err) {
    console.log("Caught:", err.message);
  }
}
main();

In [ ]:
# ===== Option A: Google Colab (active) =====
!node async_demo.js

# ===== Option B: VS Code =====
# node async_demo.js

### TODO 2: Use `Promise.allSettled` with one dish that fails, and print the status of each result.

In [ ]:
# TODO: write your code here (use %%writefile and !node)

## 8. JSON: parse and stringify

In [ ]:
%%writefile json_demo.js
const ticket = { id: 7, title: "Slow 5G data", tags: ["5g", "speed"], customer: { name: "Mei" } };

const text = JSON.stringify(ticket);
console.log(text, "|", typeof text);
console.log(JSON.stringify(ticket, null, 2));

const back = JSON.parse(text);
console.log(back.customer.name, back.tags.length);

try {
  JSON.parse("{bad json}");
} catch (e) {
  console.log("Invalid JSON:", e.name);
}

In [ ]:
# ===== Option A: Google Colab (active) =====
!node json_demo.js

# ===== Option B: VS Code =====
# node json_demo.js

## 9. fetch: an async client for a ticket API (SmartDesk step 7)
To keep this notebook self-contained, the script starts a tiny ticket API with Node's built-in `http` module, then an async client calls it with `fetch`. On your laptop, point `BASE` at your SmartDesk FastAPI instead (for example `http://127.0.0.1:8000`).

In [ ]:
%%writefile client.js
const http = require("http");
const tickets = [
  { id: 1, title: "No signal at home", priority: "high", status: "open" },
  { id: 2, title: "Double charge", priority: "high", status: "open" },
  { id: 3, title: "Change plan", priority: "low", status: "closed" },
  { id: 4, title: "Slow 5G data", priority: "medium", status: "pending" },
  { id: 5, title: "SIM not activated", priority: "medium", status: "open" },
  { id: 6, title: "Roaming question", priority: "medium", status: "open" },
];

// A tiny stand-in API: GET /tickets and GET /tickets/:id
const server = http.createServer((req, res) => {
  const match = req.url.match(/^\/tickets(?:\/(\d+))?$/);
  res.setHeader("Content-Type", "application/json");
  if (!match) { res.statusCode = 404; return res.end(JSON.stringify({ error: "Route not found" })); }
  if (!match[1]) return res.end(JSON.stringify(tickets));
  const t = tickets.find(x => x.id === Number(match[1]));
  if (!t) { res.statusCode = 404; return res.end(JSON.stringify({ error: "Ticket not found" })); }
  res.end(JSON.stringify(t));
});

async function getJSON(base, path) {
  const res = await fetch(base + path);
  if (!res.ok) throw new Error(`HTTP ${res.status} for ${path}`);
  return res.json();
}

server.listen(0, async () => {
  const BASE = process.env.BASE_URL || `http://127.0.0.1:${server.address().port}`;
  try {
    const all = await getJSON(BASE, "/tickets");
    console.log("Loaded", all.length, "tickets");
    const byPriority = all.reduce((acc, t) => ({ ...acc, [t.priority]: (acc[t.priority] || 0) + 1 }), {});
    console.log(byPriority);
    const [first, second] = await Promise.all([getJSON(BASE, "/tickets/1"), getJSON(BASE, "/tickets/2")]);
    console.log("In parallel:", first.title, "|", second.title);
    console.log(JSON.stringify(first));
    await getJSON(BASE, "/tickets/999");
  } catch (err) {
    console.log("Error:", err.message);
  } finally {
    server.close();
  }
});

In [ ]:
# ===== Option A: Google Colab (active) =====
!node client.js

# ===== Option B: VS Code =====
# node client.js
# Or, with your SmartDesk API running:  BASE_URL=http://127.0.0.1:8000 node client.js

### TODO 3: Add a function that prints the open tickets as a numbered list, sorted by priority (high first).

In [ ]:
# TODO: write your code here (use %%writefile and !node)

## 10. Check yourself
1. Why did the reminder in section 4 print last?
2. Why was Promise.all faster in section 7?
3. What would happen in section 9 without the `res.ok` check?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)